**superviser **

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.linear_model import Lasso
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

Data collection and preprocessing


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
#loade data
cal_data=pd.read_csv('/content/Fitbit_dataset.csv')

In [ ]:
cal_data = cal_data.drop(columns='Unnamed: 0', axis=1)

In [ ]:
cal_data.head()

In [ ]:
cal_data.isnull().sum()

In [ ]:
cal_data.info()

In [ ]:
cal_data.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode 'Gender' column
encode = LabelEncoder()
cal_data['Gender'] = encode.fit_transform(cal_data['Gender'])
cal_data['Workout_Type'] = encode.fit_transform(cal_data['Workout_Type'])

In [ ]:
cal_data.head()

In [ ]:
#standard scalar
scalar=StandardScaler()
final_data=scalar.fit_transform(cal_data)

In [ ]:
final_data=pd.DataFrame(final_data,columns=['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM',
       'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)', 'Workout_Type',
       'Fat_Percentage', 'Water_Intake (liters)',
       'Workout_Frequency (days/week)', 'Experience_Level', 'BMI', 'Base_MET',
       'HR_Intensity', 'Effective_MET', 'Calories_Burned (kcal)'])

In [ ]:
final_data.head()

In [ ]:
x=final_data.drop(columns='Calories_Burned (kcal)',axis=1)
y=final_data['Calories_Burned (kcal)']

In [ ]:
#import train test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=3)

In [ ]:
print(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

In [ ]:
models=[LinearRegression(),Ridge(),SVR(),RandomForestRegressor(),DecisionTreeRegressor(),Lasso(),XGBRegressor()]

In [ ]:
models = [
    LinearRegression(),
    Ridge(),
    SVR(),
    RandomForestRegressor(random_state=42),
    DecisionTreeRegressor(random_state=42),
    Lasso(),
    XGBRegressor()
]

results = []

for model in models:
    model.fit(x_train, y_train)
    x_test_predict = model.predict(x_test)

    mae = mean_absolute_error(y_test, x_test_predict)
    rmse = np.sqrt(mean_squared_error(y_test, x_test_predict))
    r2 = r2_score(y_test, x_test_predict)

    results.append({
        "Model": model.__class__.__name__,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

results_df = pd.DataFrame(results)

print(results_df)

In [ ]:
results_df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))

sns.barplot(x="Model", y="MAE", data=results_df, ax=axes[0])
axes[0].set_title("MAE")
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(x="Model", y="RMSE", data=results_df, ax=axes[1])
axes[1].set_title("RMSE")
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(x="Model", y="R2 Score", data=results_df, ax=axes[2])
axes[2].set_title("R² Score")
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

unsupervised model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from sklearn.cluster import KMeans

In [ ]:
#corrected version
cluster_data = final_data.drop(
    columns=['Workout_Type', 'Calories_Burned (kcal)']
)

In [ ]:
pca = PCA(n_components=0.95, random_state=42)

principal_components = pca.fit_transform(cluster_data)

In [ ]:
wcss = []

for i in range(2,11):

    kmeans = KMeans(
        n_clusters=i,
        init='k-means++',
        random_state=42,
        n_init=10
    )

    kmeans.fit(principal_components)

    wcss.append(kmeans.inertia_)

In [ ]:
# plot an elbow graph

sns.set()
plt.plot(range(2,11), wcss)
plt.title('The Elbow Point Graph')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

Y = kmeans.fit_predict(principal_components)

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    x=principal_components[:,0],
    y=principal_components[:,1],
    hue=Y,
    palette='viridis',
    s=60
)

plt.scatter(
    kmeans.cluster_centers_[:,0],
    kmeans.cluster_centers_[:,1],
    marker='X',
    s=200,
    color='red',
    label='Centroids'
)

plt.title("Workout Pattern Clusters")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.legend()

plt.show()

In [ ]:
score = silhouette_score(principal_components, Y)

print("Silhouette Score:", round(score,3))

In [ ]:
if score >= 0.15:
    print("Accepted")
else:
    print("Not Accepted")

In [ ]:
final_data['Cluster'] = Y

In [ ]:
cluster_summary = final_data.groupby("Cluster").mean(numeric_only=True)

cluster_summary=pd.DataFrame(cluster_summary)
cluster_summary.head()

In [ ]:
plt.figure(figsize=(12,5))

sns.heatmap(
    cluster_summary,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Cluster Feature Means")

plt.show()

In [ ]:
pip install streamlit